In [47]:
import spacy
import bm25s

In [48]:

nlp = spacy.load("da_core_news_sm")

def tokenize_passage(passage: str) -> list[str]:
    return [t.lemma_.lower() for t in nlp(passage) if not t.is_stop and not t.is_punct]
def tokenize_passages(passages: list[str]) -> list[list[str]]:
    return [tokenize_passage(p) for p in passages]


In [49]:
# load corpus
import json
from lcr.config import PROCESSED_DATA_DIR
data_dir = PROCESSED_DATA_DIR / "danish_final_cluster_r10" / "merged"
metadata_corpus = []

with open(data_dir / "chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        metadata_corpus.append(json.loads(line))
text_corpus = [entry["chunk"] for entry in metadata_corpus]
text_corpus_tokenized = tokenize_passages(text_corpus)

In [50]:
retriever = bm25s.BM25(corpus=metadata_corpus)
retriever.index(text_corpus_tokenized)

In [51]:
# load queries
contextual_queries = []
with open(data_dir / "contextual_queries.jsonl", "r", encoding="utf-8") as f:
    contextual_queries = [json.loads(line) for line in f]
tokenized_queries = tokenize_passages([entry["query"] for entry in contextual_queries])

In [54]:
results, score = retriever.retrieve(tokenized_queries, k=10, return_as='tuple', show_progress=True)

BM25S Retrieve:   0%|          | 0/162 [00:00<?, ?it/s]

In [55]:
# count how many of the retrieved results were right; how many of them were in the top 10
recal_at_1 = 0
recal_at_10 = 0
for contextual_query, retrieved in zip(contextual_queries, results):
    relevant_chunk_id = contextual_query["chunk_id"]
    retrieved_chunk_ids = [entry["chunk_id"] for entry in retrieved]
    if relevant_chunk_id in retrieved_chunk_ids:
        recal_at_10 += 1
        if retrieved_chunk_ids[0] == relevant_chunk_id:
            recal_at_1 += 1
print(f"Recall@1: {recal_at_1 / len(contextual_queries):.4f}")
print(f"Recall@10: {recal_at_10 / len(contextual_queries):.4f}")



Recall@1: 0.0062
Recall@10: 0.1975


In [43]:
# test of my retriever
from datasets.arrow_dataset import Dataset
from lcr.formatter import DataFormatter
from lcr.modeling.bm25_embedder import BM25Embedder
n_retr = BM25Embedder(spacy_model="da_core_news_sm")
data_formatter = DataFormatter()
data_formatter.load_from_jsonl(data_dir / "contextual_queries.jsonl", query_or_dataset="queries")
data_formatter.load_from_jsonl(data_dir / "chunks.jsonl", query_or_dataset="documents")


Loaded queries dataset from jsonl /Users/m-proj/repositories/long-context-retrieval/data/processed/danish_final_cluster_r10/merged/contextual_queries.jsonl
Loaded documents dataset from jsonl /Users/m-proj/repositories/long-context-retrieval/data/processed/danish_final_cluster_r10/merged/chunks.jsonl


In [44]:

results, gt_chunk_ids, metrics = n_retr.compute_results(data_formatter)


In [46]:
metrics

{'recall_at_1': 0.00617,
 'recall_at_5': 0.12346,
 'recall_at_10': 0.19753,
 'recall_at_50': 0.51235,
 'recall_at_100': 0.62346,
 'precision_at_1': 0.00617,
 'precision_at_5': 0.02469,
 'precision_at_10': 0.01975,
 'precision_at_50': 0.01025,
 'precision_at_100': 0.00623,
 'mrr_at_1': 0.006172839506172839,
 'mrr_at_5': 0.04331275720164608,
 'mrr_at_10': 0.05291250244953948,
 'mrr_at_50': 0.07115759971940097,
 'mrr_at_100': 0.0726853467397356}

In [7]:
for x in zip(results,score):

    print(x)
    break

(array([{'chunk': '§ 91. Landsbyggefonden kan af de midler, der er overført til landsdispositionsfonden, jf. § 89, give tilsagn om ydelsesstøtte til almene boligafdelinger til lån til finansiering af:\n1) Ekstraordinære udbedrings- og opretningsarbejder og afhjælpning af sundhedsskadelige forhold inden for en samlet investeringsramme på 8.080 mio. kr. i årene 2021-2026.\n2) Tilgængelighedsarbejder inden for en samlet investeringsramme på 1.641 mio. kr. i årene 2021-2026.\n3) Ombygning og sammenlægning af lejligheder inden for en samlet investeringsramme på 303 mio. kr. i årene 2021-2026.\n4) Forbedring af fællesarealer inden for en samlet investeringsramme på 965 mio. kr. i årene 2021-2026.', 'chunk_id': 'almenboligloven_§91_stk1', 'chunk_idx': 474, 'implicit_context_chunks': ['almenboligloven_§91_stk10', 'almenboligloven_§91_stk11', 'almenboligloven_§91_stk12', 'almenboligloven_§91_stk13', 'almenboligloven_§91_stk14', 'almenboligloven_§91_stk15', 'almenboligloven_§91_stk16', 'almenbol

In [92]:
for x in results:
    print(len(x))

10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10


In [28]:
results

array([[0, 1, 2, 3]])

In [34]:
text_corpus = [
    "a cat is a feline and likes to purr",
    "a dog is the human's best friend and loves to play",
]

metadata_corpus = [
    {"id": "cat-doc", "title": "About Cat", "text": text_corpus[0]},
    {"id": "dog-doc", "title": "About Dog", "text": text_corpus[1]},
]




# Pick the text field(s) you want to index.
# corpus_tokens = bm25s.tokenize([doc["text"] for doc in metadata_corpus])
# corpus_tokens


stemmer = None
stopwords = ["is"]
splitter = lambda x: x.split() # function or regex pattern
# Create a tokenizer
tokenizer = Tokenizer(
      stemmer=stemmer, stopwords=stopwords, splitter=splitter
)

corpus_tokens = tokenizer.tokenize(text_corpus)

# Retrieved documents will be the original dictionary entries.
retriever = bm25s.BM25(corpus=metadata_corpus)
retriever.index(corpus_tokens)

# Tests

In [8]:
# You can provide a list of queries instead of a single query
queries = ["What is a cat?", "is the bird a dog?"]

# Provide your own stopwords list if you don't like the default one
stopwords = ["a", "the"]

# For stemming, use any function that is callable on each word list
stemmer_fn = lambda lst: [word for word in lst]

# Tokenize the queries
query_token_ids = bm25s.tokenize(queries, stopwords=stopwords, stemmer=stemmer_fn)

# If you want the tokenizer to return strings instead of token ids, you can do this
query_token_strs = bm25s.tokenize(queries, return_ids=False)

# You can use a different corpus for retrieval, e.g., titles instead of full docs
titles = ["About Cat", "About Dog", "About Bird", "About Fish"]

# You can also choose to only return the documents and omit the scores
# note: if you pass a new corpus here, it must have the same length as your indexed corpus
results = retriever.retrieve(query_token_ids, corpus=titles, k=2, return_as="documents")

# The documents are returned as a numpy array of shape (n_queries, k)
for i in range(results.shape[1]):
    print(f"Rank {i+1}: {results[0, i]}")

IndexError: list index out of range

In [27]:
import bm25s

# Create your corpus here
corpus = [
    "a cat is a feline and likes to purr",
    "a dog is the human's best friend and loves to play",
    "a bird is a beautiful animal that can fly",
    "a fish is a creature that lives in water and swims",
]

# optional: create a stemmer

# Tokenize the corpus and only keep the ids (faster and saves memory)
corpus_tokens = bm25s.tokenize(corpus, stopwords="en", stemmer=None)

# Create the BM25 model and index the corpus
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# Query the corpus
query = "who did this?"
query_tokens = bm25s.tokenize(query, stemmer=None)

# Get top-k results as a tuple of (doc ids, scores). Both are arrays of shape (n_queries, k).
# To return docs instead of IDs, set the `corpus=corpus` parameter.
results, scores = retriever.retrieve(query_tokens, k=4, sorted=False)

for i in range(results.shape[1]):
    doc, score = results[0, i], scores[0, i]
    print(f"Rank {i+1} (score: {score:.2f}): {doc}")
results

Rank 1 (score: 0.00): 0
Rank 2 (score: 0.00): 1
Rank 3 (score: 0.00): 2
Rank 4 (score: 0.00): 3


array([[0, 1, 2, 3]])

array([[1, 0, 2, 3]])

In [18]:
scores

array([[0.        , 1.0584376 , 0.        , 0.48158914]], dtype=float32)

In [31]:
for t in nlp("xd haha"):
    print(t, t.lemma_, t.is_alpha, t.is_stop)

xd xd True False
haha haha True False


In [32]:
doc = nlp("jf. lovbekendtgørelse nr. 123")
print([t.text for t in doc])
# ['jf.', 'lovbekendtgørelse', 'nr.', '123']

['jf.', 'lovbekendtgørelse', 'nr.', '123']


In [26]:

from bm25s.tokenization import Tokenizer

corpus = [
      "b",
      "i",
      "a cat is a feline and likes to purr",
      "a dog is the human's best friend and loves to play",
      "a bird is a beautiful animal that can fly",
      "a fish is a creature that lives in water and swims",
]

# Pick your favorite stemmer, and pass 
# stemmer = None
def dummy_stemmer(word: list[str]) -> list[str]:
      print(word)
      return word
# Create a tokenizer
tokenizer = Tokenizer(
      stemmer=dummy_stemmer, stopwords=[]
)
corpus_tokens1 = tokenizer.tokenize(corpus)

# corpus_tokens2 = bm25s.tokenize(corpus, stemmer=dummy_stemmer, stopwords=[""])
# corpus_tokens

cat
is
feline
and
likes
to
purr
dog
the
human
best
friend
loves
play
bird
beautiful
animal
that
can
fly
fish
creature
lives
in
water
swims


In [15]:
corpus_tokens2

Tokenized(
  "ids": [
    0: [15, 7, 17, 0, 9, 11, 22]
    1: [24, 7, 23, 16, 14, 1, 0, 20, 11, 25]
    2: [5, 7, 4, 8, 18, 19, 3]
    3: [2, 7, 6, 18, 12, 13, 21, 0, 10]
  ],
  "vocab": [
    'and': 0
    'animal': 8
    'beautiful': 4
    'best': 14
    'bird': 5
    'can': 19
    'cat': 15
    'creature': 6
    'dog': 24
    'feline': 17
    ... (total 26 tokens)
  ],
)

In [14]:
corpus_tokens1

[[1, 2, 3, 4, 5, 6, 7],
 [8, 2, 9, 10, 11, 12, 4, 13, 6, 14],
 [15, 2, 16, 17, 18, 19, 20],
 [21, 2, 22, 18, 23, 24, 25, 4, 26]]

In [ ]:

# let's see what the tokens look like
print("tokens:", corpus_tokens)
print("vocab:", tokenizer.get_vocab_dict())

# note: the vocab dict will either be a dict of `word -> id` if you don't have a stemmer, and a dict of `stemmed word -> stem id` if you do.
# You can save the vocab. it's fine to use the same dir as your index if filename doesn't conflict
tokenizer.save_vocab(save_dir="bm25s_very_big_index")

# loading:
new_tokenizer = Tokenizer(stemmer=stemmer, stopwords=[], splitter=splitter)
new_tokenizer.load_vocab("bm25s_very_big_index")
print("vocab reloaded:", new_tokenizer.get_vocab_dict())

# the same can be done for stopwords
print("stopwords before reload:", new_tokenizer.stopwords)
tokenizer.save_stopwords(save_dir="bm25s_very_big_index")
new_tokenizer.load_stopwords("bm25s_very_big_index")
print("stopwords reloaded:", new_tokenizer.stopwords)

In [27]:
import spacy
from spacy.lang.da.examples import sentences 

nlp = spacy.load("da_core_news_sm")
doc = nlp(sentences[0])
print(doc.text)
for token in doc:
    print(token.text, token.pos_, token.dep_)

Apple overvejer at købe et britisk startup for 1 milliard dollar.
Apple PROPN nsubj
overvejer VERB ROOT
at PART mark
købe VERB obj
et DET det
britisk ADJ amod
startup NOUN obj
for ADP case
1 NUM nummod
milliard NOUN nmod
dollar ADJ nmod
. PUNCT punct


In [28]:
sentences[0]

'Apple overvejer at købe et britisk startup for 1 milliard dollar.'

In [28]:

corpus_tokens = bm25s.tokenize(text_corpus)


In [29]:
corpus_tokens

Tokenized(
  "ids": [
    0: [0, 1, 2, 3, 4, 5]
    1: [6, 7, 2, 3, 4, 5, 8, 9, 10, 11, ...]
    2: [37]
    3: [38, 39]
    4: [40]
    5: [41, 38, 42, 43, 41, 42, 17, 44, 1, 45, ...]
    6: [47, 82, 83, 41, 62, 63, 84, 5, 17, 43, ...]
    7: [92, 47, 82, 83, 93, 17, 94, 43, 85, 14, ...]
    8: [92, 47, 82, 83, 41, 62, 64, 84, 5, 17, ...]
    9: [92, 47, 82, 83, 41, 62, 121, 84, 66, 122, ...]
    ... (total 7743 docs)
  ],
  "vocab": [
    '00': 5765
    '000': 1579
    '01': 2774
    '020': 2766
    '03': 2776
    '055': 4422
    '080': 2486
    '10': 230
    '100': 1203
    '101': 814
    ... (total 10885 tokens)
  ],
)

In [ ]:

retriever = bm25s.BM25(corpus=metadata_corpus)
retriever.index(corpus_tokens)

In [ ]:
## repeat for polish